# 12 · Representações contextuais: uma palavra, muitos vetores (Peters et al., 2018 — ELMo)

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


**Artigo-base (leitura complementar):** Peters, Neumann, Iyyer, Gardner, Clark, Lee & Zettlemoyer (2018), *Deep contextualized word representations* (NAACL).

## O limite de tudo o que vimos até aqui
word2vec, GloVe, fastText e Doc2Vec dão **um vetor por tipo**: *banco* (assento) e *banco* (instituição) compartilham o mesmo ponto; *manga* (fruta/camisa) idem. Faruqui et al. (2016) apontam a polissemia como limite da avaliação intrínseca; Bolukbasi et al. notam que *man* é ambíguo (§5.1).

## A proposta ELMo (*Embeddings from Language Models*)
1. Treinar um **modelo de linguagem bidirecional** (biLM): um LSTM *forward* ($p(t_k\mid t_1..t_{k-1})$) e um *backward* ($p(t_k\mid t_{k+1}..t_N)$), com $L=2$ camadas, entrada por **CNN de caracteres** (sem OOV), parâmetros de entrada e softmax compartilhados entre as direções (§3.1).
2. Para cada token, expor **todas as camadas**: $R_k = \{x_k, \overrightarrow{h}_{k,j}, \overleftarrow{h}_{k,j}\}$, $2L+1$ vetores (§3.2).
3. A tarefa aprende uma **mistura**:
$$\mathrm{ELMo}_k^{task} = \gamma^{task} \sum_{j=0}^{L} s_j^{task}\, h_{k,j}^{LM} \qquad (1)$$
com $s$ *softmax-normalizado* e $\gamma$ um escalar.
4. Concatenar $[x_k; \mathrm{ELMo}_k]$ à entrada (e às vezes à saída) de um modelo supervisionado existente, com o biLM **congelado** (§3.3).

Resultados: estado da arte em 6 tarefas (SQuAD +4,7 F1, SNLI, SRL +3,2, coref +3,2, NER, SST-5 +3,3) com reduções relativas de erro de 6–20 % (Tabela 1). Análise (§5.3): camadas baixas codificam **sintaxe** (POS), camadas altas codificam **sentido** (WSD).

## O que fazemos aqui
Um **ELMo-lite** em JAX (`replang.models.bilm_jax`): biLM de 2 camadas LSTM (128 unidades), *lookup* de palavras (sem CNN de caracteres), treinado por alguns minutos em Machado de Assis (`replang train bilm`). Suficiente para **mostrar** o mecanismo: (a) vetores diferentes por contexto; (b) camadas diferentes servem a tarefas diferentes; (c) a mistura da eq. (1).

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: Y:\BLIS\linguagemBLIS2S26\linguagem-blis2s26 | fast: False


In [2]:
from replang.models.bilm_jax import HAS_JAX, BiLM, BiLMVocab, elmo_mix, contextual_vectors
assert HAS_JAX, "instale o extra: uv sync --extra contextual"
import json
model, vocab = BiLM.load(PATHS.models / "machado_bilm.pkl")
meta = json.loads((PATHS.models / "machado_bilm.json").read_text())
print(f"biLM: {meta['layers']} camadas × {meta['hid']} unidades, embedding {meta['emb']}d, vocabulário {meta['vocab']:,}, {meta['steps']} passos")
from replang.viz import training_curve
training_curve(model.history, x="step", y="ppl", title="Perplexidade de treino do biLM (média das direções)").show()

biLM: 2 camadas × 128 unidades, embedding 64d, vocabulário 8,004, 2500 passos


### Arquitetura em código

O coração do modelo é o `scan` sobre a sequência com uma célula LSTM, uma vez para a frente e outra para trás, por camada; a saída tem forma `(L+1, T, 2·hid)` — a camada 0 é o embedding do token duplicado, como no artigo.

In [3]:
import inspect
from replang.models import bilm_jax
print(inspect.getsource(bilm_jax._lstm_scan))
src = inspect.getsource(BiLM._forward_all); print(src)

def _lstm_scan(params, xs, hid):
    """xs: (T, B, in) → hs: (T, B, hid)."""

    def step(carry, x):
        h, c = carry
        z = x @ params["Wx"] + h @ params["Wh"] + params["b"]
        i, f, g, o = jnp.split(z, 4, axis=-1)
        c = jax.nn.sigmoid(f) * c + jax.nn.sigmoid(i) * jnp.tanh(g)
        h = jax.nn.sigmoid(o) * jnp.tanh(c)
        return (h, c), h

    B = xs.shape[1]
    init = (jnp.zeros((B, hid)), jnp.zeros((B, hid)))
    _, hs = jax.lax.scan(step, init, xs)
    return hs

    def _forward_all(self, params, tokens):
        """tokens: (B, T) → camadas (L+1, B, T, 2*hid) e logits fwd/bwd (B, T, V)."""
        x = params["E"][tokens]  # (B, T, emb)
        x = jnp.tanh(x @ params["proj_in"])  # (B, T, hid)
        xs = jnp.swapaxes(x, 0, 1)  # (T, B, hid)
        h_f, h_b = xs, xs[::-1]
        layers_f, layers_b = [], []
        for j in range(self.L):
            out_f = _lstm_scan(params["fwd"][j], h_f, self.hid)
            out_b = _lstm_scan(params["bwd"][j], h_

## 1. O modelo de linguagem funciona? Perplexidade e previsões

Um biLM minúsculo treinado por minutos não é o `CNN-BIG-LSTM` do artigo (perplexidade 39,7 no 1B Word Benchmark). Mas deve ter aprendido regularidades: artigos antes de substantivos, concordância, colocações de Machado.

In [4]:
import jax, jax.numpy as jnp
from replang.data.corpora import load_machado_sentences
sents = load_machado_sentences()
held = [vocab.encode(s) for s in sents[-300:]]
print(f"perplexidade em 300 sentenças finais (não vistas se o treino amostrou do início): {model.perplexity(held):.1f}  (uniforme seria {len(vocab):,})")
def top_next(prefix, k=6):
    enc = vocab.encode(prefix)[:-1]   # sem </s>
    _, lf, _ = model._fwd_fn(model.params, jnp.asarray(np.array(enc, dtype=np.int32)[None]))
    p = np.asarray(jax.nn.softmax(lf[0, -1]))
    return [(vocab.words[i], round(float(p[i]), 3)) for i in np.argsort(-p)[:k]]
for prefix in [["capitu", "olhou", "para"], ["o", "senhor"], ["não", "é"], ["a", "casa", "de"]]:
    print(" ".join(prefix), "→", top_next(prefix))

perplexidade em 300 sentenças finais (não vistas se o treino amostrou do início): 182.8  (uniforme seria 8,004)


capitu olhou para → [('o', 0.181), ('a', 0.166), ('ele', 0.126), ('ela', 0.125), ('<unk>', 0.062), ('casa', 0.035)]
o senhor → [('é', 0.082), ('<unk>', 0.045), ('não', 0.039), ('era', 0.032), ('que', 0.019), ('foi', 0.013)]
não é → [('preciso', 0.139), ('o', 0.043), ('que', 0.042), ('<unk>', 0.033), ('a', 0.033), ('verdade', 0.022)]
a casa de → [('maria', 0.043), ('<unk>', 0.042), ('um', 0.034), ('luís', 0.031), ('d', 0.031), ('uma', 0.025)]


## 2. Uma palavra, vários vetores: *banco*, *manga*, *letra*

Construímos sentenças com dois sentidos de cada palavra e comparamos o cosseno entre os vetores contextuais do mesmo token, por camada. Na **camada 0** todos são idênticos (cosseno 1 — é o embedding estático); nas camadas LSTM o vetor passa a depender da sentença.

> **Leia os números com honestidade.** Com um biLM de 128 unidades treinado por minutos em 2,4 M tokens, a *dependência do contexto* aparece claramente (cossenos caem de 1,00 para ~0,8–0,9), mas a *separação de sentidos* é marginal (diferença de poucos centésimos entre "mesmo sentido" e "sentidos diferentes"). O ELMo original tem 4096 unidades, CNN de caracteres e 1 bilhão de tokens; é essa escala que faz a Tabela 4 do artigo funcionar. O mecanismo é o mesmo — a capacidade, não.

In [5]:
from replang.utils import tokenize
exemplos = {
    "banco": ["ele sentou no banco da praça para descansar", "o velho dormia no banco de madeira do jardim", "ela foi ao banco sacar o dinheiro da conta", "o gerente do banco negou o empréstimo ao comerciante"],
    "manga": ["a manga da camisa estava rasgada e suja", "ele dobrou a manga do casaco até o cotovelo", "comi uma manga madura e doce no quintal", "a manga é uma fruta tropical muito saborosa"],
    "letra": ["a letra da música falava de amor e saudade", "ele cantou a letra inteira da canção", "a letra dela era bonita e redonda no papel", "escreveu a carta com letra firme e clara"],
}
def sim_matrix(word, frases, layer):
    V = []
    for f in frases:
        toks = tokenize(f)
        V.append(contextual_vectors(model, vocab, toks, word, layer=layer))
    V = np.stack(V); V /= np.linalg.norm(V, axis=1, keepdims=True)
    return V @ V.T
for word, frases in exemplos.items():
    print(f"\n=== {word} ===  (sentenças 1-2: sentido A; 3-4: sentido B)")
    for layer in [0, 1, 2]:
        S = sim_matrix(word, frases, layer)
        same = np.mean([S[0, 1], S[2, 3]]); cross = np.mean([S[0, 2], S[0, 3], S[1, 2], S[1, 3]])
        print(f"  camada {layer}: cos(mesmo sentido) = {same:.2f} | cos(sentidos diferentes) = {cross:.2f} | diferença = {same - cross:+.2f}")


=== banco ===  (sentenças 1-2: sentido A; 3-4: sentido B)


  camada 0: cos(mesmo sentido) = 1.00 | cos(sentidos diferentes) = 1.00 | diferença = +0.00
  camada 1: cos(mesmo sentido) = 0.92 | cos(sentidos diferentes) = 0.86 | diferença = +0.06
  camada 2: cos(mesmo sentido) = 0.92 | cos(sentidos diferentes) = 0.86 | diferença = +0.06

=== manga ===  (sentenças 1-2: sentido A; 3-4: sentido B)
  camada 0: cos(mesmo sentido) = 1.00 | cos(sentidos diferentes) = 1.00 | diferença = +0.00
  camada 1: cos(mesmo sentido) = 0.79 | cos(sentidos diferentes) = 0.79 | diferença = -0.00
  camada 2: cos(mesmo sentido) = 0.80 | cos(sentidos diferentes) = 0.79 | diferença = +0.01

=== letra ===  (sentenças 1-2: sentido A; 3-4: sentido B)
  camada 0: cos(mesmo sentido) = 1.00 | cos(sentidos diferentes) = 1.00 | diferença = +0.00
  camada 1: cos(mesmo sentido) = 0.72 | cos(sentidos diferentes) = 0.81 | diferença = -0.09
  camada 2: cos(mesmo sentido) = 0.74 | cos(sentidos diferentes) = 0.82 | diferença = -0.08


In [6]:
import plotly.express as px
word, frases = "banco", exemplos["banco"]
S = sim_matrix(word, frases, 2)
labels = [f"s{i+1}: " + " ".join(tokenize(f))[:38] for i, f in enumerate(frases)]
px.imshow(S, x=[f"s{i+1}" for i in range(4)], y=labels, zmin=0, zmax=1, color_continuous_scale="Blues", text_auto=".2f", template="plotly_white",
          title="cos entre os vetores contextuais de 'banco' (camada 2)").show()

## 3. Vizinhos contextuais (Tabela 4 do artigo)

O artigo compara os vizinhos de *play* no GloVe (misturam esporte e teatro) com os vizinhos do biLM, que trazem **tokens em contexto** do mesmo sentido. Fazemos o mesmo: para cada ocorrência-alvo, buscamos entre os tokens de um conjunto de sentenças de Machado os mais parecidos.

In [7]:
from replang.data.embeddings import load_local_model
sg = load_local_model("machado_sg100")
print("vizinhos ESTÁTICOS de 'banco' (Skip-gram):", [w for w, _ in sg.most_similar("banco", topn=10)])
pool = [s for s in sents if "banco" in s and len(s) <= 25][:400]
tok_vecs, tok_info = [], []
for s in pool:
    reps = model.representations(vocab.encode(s))
    for i, w in enumerate(s):
        if w == "banco":
            tok_vecs.append(reps[2, i + 1]); tok_info.append(" ".join(s))
T = np.stack(tok_vecs); T /= np.linalg.norm(T, axis=1, keepdims=True)
for f in [frases[0], frases[2]]:
    q = contextual_vectors(model, vocab, tokenize(f), "banco", layer=2); q /= np.linalg.norm(q)
    sims = T @ q
    print(f"\nalvo: «{f}»")
    for i in np.argsort(-sims)[:5]:
        print(f"   {sims[i]:.2f}  {tok_info[i][:110]}")

vizinhos ESTÁTICOS de 'banco' (Skip-gram):

 ['beco', 'depósito', 'clube', 'mato', 'arsenal', 'laemmert', 'respectivo', 'barco', 'batalhão', 'tomé']



alvo: «ele sentou no banco da praça para descansar»
   0.95  nesse mesmo dia levei-os ao banco do brasil
   0.94  de repente ouvi vozes estranhas pareceu-me que eram os burros que conversavam inclinei-me ia no banco da frent
   0.94  afonso pena presidente do banco da república
   0.94  parou no ângulo onde estava a malva silvestre tornou a ver a pedra onde costumava sentar-se tornou a ver o ban
   0.93  entro ponho o chapéu no banco e sento-me em cima

alvo: «ela foi ao banco sacar o dinheiro da conta»
   0.95  recolhi-me ao banco soluçando fustigado pelos impropérios do mestre
   0.94  queira sentar-se aí no banco ou se prefere entrar
   0.94  dondon saiu o diretor do banco despedia-se palha agradecia-lhe a fineza estimava-lhe a saúde
   0.93  vou passar pelo banco antes de entrar em casa
   0.93  não foi nada papai disse iaiá lendo no rosto do pai o motivo que o trouxera fui pular do banco e caí


## 4. Que camada serve a que tarefa? (§5.3: POS × WSD)

O artigo mede, com classificadores lineares sobre representações **congeladas**: POS tagging é melhor com a **1ª camada** (97,3 vs 96,8), desambiguação de sentido com a **2ª** (69,0 vs 67,4). Reproduzimos o lado da sintaxe com o Mac-Morpho: regressão logística sobre a representação de cada camada (e sobre o Skip-gram estático).

In [8]:
from replang.data.corpora import load_macmorpho
from replang.eval.linear import LinearClassifier
from replang.accel import report
print('aceleração:', report())
train = load_macmorpho(split="train", limit_sentences=1200 if FAST else 2500); test = load_macmorpho(split="test", limit_sentences=400)
# representações de todas as camadas, calculadas UMA vez por sentença (sem <s> e </s>)
def all_layers(sent_tags):
    toks = [w.lower() for w, _ in sent_tags]
    return model.representations(vocab.encode(toks))[:, 1:-1]
import time; t = time.time()
reps_train = [all_layers(s) for s in train]; reps_test = [all_layers(s) for s in test]
ytr = [t_ for s in train for _, t_ in s]; yte = np.array([t_ for s in test for _, t_ in s])
print(f"representações de {len(train)+len(test)} sentenças em {time.time()-t:.0f}s")
rows = []
for layer in [0, 1, 2]:
    Xtr = np.vstack([r[layer] for r in reps_train]); Xte = np.vstack([r[layer] for r in reps_test])
    clf = LinearClassifier(max_iter=400, C=1.0).fit(Xtr, ytr)
    rows.append({"representação": f"biLM camada {layer}", "dim": Xtr.shape[1], "acurácia POS (linear, sem janela)": float((clf.predict(Xte) == yte).mean())})
static = lambda s: np.stack([sg.get(w.lower(), np.zeros(sg.dim)) for w, _ in s])
Xtr = np.vstack([static(s) for s in train]); Xte = np.vstack([static(s) for s in test])
clf = LinearClassifier(max_iter=400).fit(Xtr, ytr)
rows.append({"representação": "Skip-gram estático (sem janela)", "dim": sg.dim, "acurácia POS (linear, sem janela)": float((clf.predict(Xte) == yte).mean())})
pos_layers = pd.DataFrame(rows)
pos_layers.style.format({"acurácia POS (linear, sem janela)": "{:.3f}"}).background_gradient(subset=["acurácia POS (linear, sem janela)"], cmap="Greens")

aceleração: backend=jax · dispositivo=CPU (20 threads) · jobs=5


representações de 2900 sentenças em 10s


,representação,dim,"acurácia POS (linear, sem janela)"
0,biLM camada 0,256,0.635
1,biLM camada 1,256,0.784
2,biLM camada 2,256,0.782
3,Skip-gram estático (sem janela),100,0.637


Sem janela de contexto, o vetor estático "não sabe" se *a* é artigo ou preposição; o biLM sabe, porque o vetor do token já viu a sentença inteira. É a diferença entre **representar o tipo** e **representar o token**.

## 5. A mistura ELMo: aprendendo $s^{task}$ (eq. 1, Fig. 2)

Em vez de escolher uma camada, a tarefa aprende pesos. Fazemos uma busca simples sobre $s$ (softmax de 3 parâmetros) para a tarefa de POS e para a tarefa de "separar sentidos" (maximizar a diferença intra/inter-sentido da seção 2). O artigo observa: tarefas sintáticas pesam a camada 1; semânticas, a camada 2 (Fig. 2). No nosso modelo pequeno, espere o resultado de POS ser nítido (pesos nas camadas LSTM) e o de sentidos ser inconclusivo (a diferença da seção 2 é ≈ 0 em todas as camadas) — o que, por si, já ilustra a Fig. 2: *os pesos refletem onde a informação útil está*.

In [9]:
from itertools import product
grid = [np.array(w) for w in product([0.0, 1.0, 2.0], repeat=3)]
def softmax(w): e = np.exp(w - w.max()); return e / e.sum()
# tarefa A: POS (subconjunto menor para rapidez)
tr, te = train[:600], test[:150]
reps_tr, reps_te = reps_train[:600], reps_test[:150]
ytr_s = [t_ for s in tr for _, t_ in s]; yte_s = np.array([t_ for s in te for _, t_ in s])
best_pos = None
for w in grid[::2]:
    s = softmax(w)
    Xtr = np.vstack([np.tensordot(s, r, axes=(0, 0)) for r in reps_tr]); Xte = np.vstack([np.tensordot(s, r, axes=(0, 0)) for r in reps_te])
    acc = float((LinearClassifier(max_iter=300).fit(Xtr, ytr_s).predict(Xte) == yte_s).mean())
    if best_pos is None or acc > best_pos[0]: best_pos = (acc, s)
# tarefa B: separação de sentidos
best_wsd = None
for w in grid:
    s = softmax(w); diffs = []
    for word, frases in exemplos.items():
        V = np.stack([elmo_mix(model.representations(vocab.encode(tokenize(f))), np.log(s + 1e-9))[tokenize(f).index(word) + 1] for f in frases])
        V /= np.linalg.norm(V, axis=1, keepdims=True); S = V @ V.T
        diffs.append(np.mean([S[0, 1], S[2, 3]]) - np.mean([S[0, 2], S[0, 3], S[1, 2], S[1, 3]]))
    score = float(np.mean(diffs))
    if best_wsd is None or score > best_wsd[0]: best_wsd = (score, s)
from replang.viz import layer_weights_figure
print(f"POS: melhor acurácia {best_pos[0]:.3f} com pesos {np.round(best_pos[1], 2)}")
print(f"separação de sentidos: melhor diferença {best_wsd[0]:+.3f} com pesos {np.round(best_wsd[1], 2)}")
layer_weights_figure({"POS tagging": best_pos[1], "separar sentidos": best_wsd[1]}, title="Pesos s^task aprendidos (busca em grade) — cf. Fig. 2 de Peters et al.").show()

POS: melhor acurácia 0.763 com pesos [0.21 0.58 0.21]
separação de sentidos: melhor diferença -0.001 com pesos [0.79 0.11 0.11]


## 6. Eficiência amostral (§5.4, Fig. 1 do artigo)

Com ELMo, o modelo SRL com **1 %** dos dados rotulados iguala o baseline com **10 %**. Reproduzimos a forma da curva no POS: acurácia × fração de sentenças rotuladas, com a camada 1 do biLM × o vetor estático.

In [10]:
rows = []
for n in ([100, 300, 1000] if FAST else [100, 300, 1000, 2500]):
    for name, Xall_tr, Xall_te in [("biLM camada 1", [r[1] for r in reps_train], [r[1] for r in reps_test]), ("Skip-gram estático", [static(s) for s in train], [static(s) for s in test])]:
        Xtr = np.vstack(Xall_tr[:n]); ytr_n = [t_ for s in train[:n] for _, t_ in s]
        Xte = np.vstack(Xall_te)
        acc = float((LinearClassifier(max_iter=300).fit(Xtr, ytr_n).predict(Xte) == yte).mean())
        rows.append({"sentenças rotuladas": n, "representação": name, "acurácia": acc})
px.line(pd.DataFrame(rows), x="sentenças rotuladas", y="acurácia", color="representação", markers=True, log_x=True, template="plotly_white", title="Eficiência amostral: POS × tamanho do treino rotulado").show()

## 7. Síntese: de word2vec a ELMo (e ao que veio depois)

| | word2vec / GloVe / fastText | ELMo |
|---|---|---|
| unidade representada | **tipo** (uma entrada de tabela) | **token** (função da sentença) |
| polissemia | um vetor para todos os sentidos | um vetor por ocorrência |
| treino | predizer vizinhos numa janela | modelo de linguagem bidirecional profundo |
| uso | tabela de consulta | rede congelada + mistura de camadas por tarefa |
| OOV | fastText: n-gramas | CNN de caracteres |

O passo seguinte (BERT, 2018) troca LSTMs por *Transformers* e a mistura de camadas por *fine-tuning* — mas a ideia-chave é a do ELMo: **o vetor é uma função do contexto**. E o viés? Continua lá, agora dependente do contexto (Zhao et al., 2019; May et al., 2019): a geometria mudou, o problema não desapareceu.

### Pontos para a apresentação
1. Mostre a matriz de cossenos de *banco*: camada 0 (tudo 1,0) × camada 2.
2. Mostre a tabela POS por camada: sem janela, o estático falha onde o contextual acerta.
3. Enfatize a **mistura aprendida** — a tarefa escolhe o que usar.

## Referências
- PETERS, M. et al. **Deep contextualized word representations.** NAACL, 2018. [arXiv:1802.05365](https://arxiv.org/abs/1802.05365)
- BOJANOWSKI, P.; GRAVE, E.; JOULIN, A.; MIKOLOV, T. **Enriching Word Vectors with Subword Information.** TACL, 2017. [arXiv:1607.04606](https://arxiv.org/abs/1607.04606)
- MIKOLOV, T.; CHEN, K.; CORRADO, G.; DEAN, J. **Efficient Estimation of Word Representations in Vector Space.** ICLR Workshop, 2013. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- ZHAO, J. et al. **Gender Bias in Contextualized Word Embeddings.** NAACL, 2019.